In [1]:
import os
import warnings
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from typing import Optional

from pinecone import Pinecone
from llama_index.core import Settings, VectorStoreIndex, PromptTemplate
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.vector_stores.pinecone import PineconeVectorStore
from llama_index.core.vector_stores import MetadataFilters, ExactMatchFilter
from llama_index.core.workflow import Workflow, StartEvent, StopEvent, step, Event

d:\Qanoon\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
warnings.filterwarnings("ignore")
load_dotenv()
Settings.llm = OpenAI(model="gpt-4o", temperature=0, api_key=os.getenv("OPENAI_API_KEY"))
Settings.embed_model = OpenAIEmbedding(model="text-embedding-3-small", api_key=os.getenv("OPENAI_API_KEY"))

In [7]:
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))
vector_store = PineconeVectorStore(pinecone_index=pc.Index("qanoon"))
index = VectorStoreIndex.from_vector_store(vector_store=vector_store)

In [4]:
class AnalyzerOutput(BaseModel):
    is_question: bool = Field(description="True إذا كان هناك سؤال قانوني، False إذا كانت مجرد تحية أو كلام عام.")
    chat_reply: str = Field(description="إذا كانت مجرد تحية، اكتب الرد هنا.")
    search_query: str = Field(description="السؤال مُصاغاً بالفصحى للبحث في قاعدة البيانات.")
    status_filter: Optional[str] = Field(description="ساري ونافذ أو ملغى (إن ذُكرت في السؤال)، وإلا None.")

class ReviewerOutput(BaseModel):
    is_approved: bool = Field(description="True إذا كانت الإجابة دقيقة ومنظمة، False إذا احتاجت تعديل.")
    critique: str = Field(description="ملاحظات للمصيغ، أو 'ممتاز' إذا قُبلت.")


class DraftEvent(Event):
    question: str
    retrieved_laws: str
    critique: str
    revision_count: int

class ReviewEvent(Event):
    question: str
    retrieved_laws: str
    draft_answer: str
    revision_count: int

In [5]:

class LegalAssistantWorkflow(Workflow):
    
    @step
    async def analyzer_and_researcher(self, ev: StartEvent) -> DraftEvent | StopEvent:
        query = ev.get("question")
    
        
        prompt = PromptTemplate("حلل رسالة المستخدم التالية: {query}")
        analysis = await Settings.llm.astructured_predict(AnalyzerOutput, prompt, query=query)
        
     
        if not analysis.is_question:
            return StopEvent(result=analysis.chat_reply)
            
        print(f"RESEARCHER '{analysis.search_query}'")
        
      
        filters = None
        if analysis.status_filter:
            filters = MetadataFilters(filters=[ExactMatchFilter(key="status", value=analysis.status_filter)])
            
        nodes = index.as_retriever(similarity_top_k=5, filters=filters).retrieve(analysis.search_query)
        
        laws_text = "\n".join([
            f"### النص {i+1} ###\nالقانون: {n.metadata.get('official_name')}\nالحالة: {n.metadata.get('status')}\nالرابط: {n.metadata.get('link')}\nالنص: {n.text}\n" 
            for i, n in enumerate(nodes)
        ])
        
        return DraftEvent(question=query, retrieved_laws=laws_text, critique="بدون ملاحظات", revision_count=0)

    @step
    async def drafter(self, ev: DraftEvent) -> ReviewEvent:
       
        print(f"DRAFTER({ev.revision_count + 1})")
        
        prompt = f"""أنت مستشار قانوني ليبي محترف وبليغ. مهمتك تقديم إجابة دقيقة ومبسطة بناءً على القوانين المرفقة فقط.
        إذا بدأ المستخدم بالتحية، بادله التحية بلباقة في جملة طبيعية قبل البدء في الإجابة.
        
        شروط الصياغة والتنسيق (إجبارية):
        1. ابدأ بمقدمة طبيعية وسلسة تذكر فيها اسم القانون وحالته (مثال: ينظم هذا الأمر قانون كذا، وهو قانون ساري ونافذ...). لا تكتب الحالة كبيانات جامدة بين أقواس.
        2. لخص النصوص القانونية الطويلة بأسلوب واضح ومفهوم للمواطن العادي، وتجنب النسخ واللصق الحرفي للفقرات الطويلة.
        3. قسّم الإجابة لنقاط مرقمة، وضع لكل نقطة عنواناً مختصراً بخط عريض (Bold).
        4. اذكر "رقم المادة" بوضوح داخل كل نقطة.
        5. أرفق الرابط في النهاية بصيغة نظيفة (مثال: للاطلاع على النص الكامل، تفضل بزيارة[الرابط]).
        6. لا تقم بتأليف أي معلومة غير موجودة في النصوص.
        
        السؤال: {ev.question}
        
        القوانين: {ev.retrieved_laws}
        
        ملاحظات التعديل (إن وجدت): {ev.critique}
        """
        
        response = await Settings.llm.acomplete(prompt)
        return ReviewEvent(question=ev.question, retrieved_laws=ev.retrieved_laws, draft_answer=str(response), revision_count=ev.revision_count + 1)
    @step
    async def reviewer(self, ev: ReviewEvent) -> DraftEvent | StopEvent:
        
        prompt = PromptTemplate(
            "أنت مراجع قانوني ليبي. هل الإجابة التالية دقيقة ومقسمة لنقاط وتذكر أرقام المواد واسم القانون والرابط بدون تأليف؟\n\n"
            "القوانين الأصلية:\n{laws}\n\nالإجابة:\n{answer}"
        )
        
        review = await Settings.llm.astructured_predict(ReviewerOutput, prompt, laws=ev.retrieved_laws, answer=ev.draft_answer)
        
        
        if review.is_approved or ev.revision_count >= 3:
            return StopEvent(result=ev.draft_answer)
            
    
        print(f" The answer was rejected. the reason: {review.critique}")
        return DraftEvent(question=ev.question, retrieved_laws=ev.retrieved_laws, critique=review.critique, revision_count=ev.revision_count)

In [11]:

legal_assistant = LegalAssistantWorkflow(timeout=120, verbose=False)

async def test_agent(query: str):
   
    print(f"User_question: {query}")
    final_answer = await legal_assistant.run(question=query)
    print("\nfinal_answer:")
    print(final_answer)

await test_agent("السلام عليكم، عندي استفسار بخصوص عضوية المجلس. لو فرضاً توفى أحد الأعضاء، ما هي الإجراءات المتبعة لملء المقعد الشاغر ومن هي الجهة المسؤولة عن تعيين البديل؟")

User_question: السلام عليكم، عندي استفسار بخصوص عضوية المجلس. لو فرضاً توفى أحد الأعضاء، ما هي الإجراءات المتبعة لملء المقعد الشاغر ومن هي الجهة المسؤولة عن تعيين البديل؟
RESEARCHER 'ما هي الإجراءات المتبعة لملء مقعد شاغر في المجلس بعد وفاة أحد الأعضاء؟ ومن هي الجهة المسؤولة عن تعيين البديل؟'
DRAFTER(1)
 The answer was rejected. the reason: الإجابة المقدمة تحتوي على بعض المعلومات الصحيحة ولكنها تفتقر إلى التنظيم الدقيق والتفصيل المطلوب. إليك بعض الملاحظات:

1. **عدم ذكر أرقام المواد واسم القانون بشكل واضح:**
   - يجب ذكر أرقام المواد واسم القانون بشكل واضح ومنظم في الإجابة.

2. **عدم وجود تفاصيل كافية حول الإجراءات:**
   - الإجابة لم تتضمن تفاصيل كافية حول الإجراءات المتعلقة بملء المقعد الشاغر بعد إسقاط العضوية.

3. **عدم الإشارة إلى جميع النصوص ذات الصلة:**
   - لم يتم الإشارة إلى جميع النصوص القانونية ذات الصلة التي قد تحتوي على معلومات إضافية حول الإجراءات.

4. **عدم وجود تقسيم واضح للنقاط:**
   - الإجابة تفتقر إلى تقسيم واضح للنقاط مما يجعلها أقل وضوحًا.

5. **الرابط المرفق:**
   -